# **MIT 6.5940 EfficientML.ai 2024秋季：实验0 PyTorch教程**

在本教程中，我们将探索如何使用PyTorch训练神经网络。

### 环境配置

我们首先安装本教程中将使用的几个包：

In [ ]:
!pip install torchprofile 1>/dev/null

然后我们将导入几个库：

In [ ]:
import random
from collections import OrderedDict, defaultdict

import numpy as np
import torch
from matplotlib import pyplot as plt
from torch import nn
from torch.optim import *
from torch.optim.lr_scheduler import *
from torch.utils.data import DataLoader
from torchprofile import profile_macs
from torchvision.datasets import *
from torchvision.transforms import *
from tqdm.auto import tqdm

为了确保可重复性，我们将控制随机数生成器的种子：

In [ ]:
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)

### 数据

在本教程中，我们将使用CIFAR-10作为目标数据集。该数据集包含10个类别的图像，每张图像的
大小为3x32x32，即3通道的32x32像素彩色图像。

In [ ]:
transforms = {
  "train": Compose([
    RandomCrop(32, padding=4),
    RandomHorizontalFlip(),
    ToTensor(),
  ]),
  "test": ToTensor(),
}

dataset = {}
for split in ["train", "test"]:
  dataset[split] = CIFAR10(
    root="data/cifar10",
    train=(split == "train"),
    download=True,
    transform=transforms[split],
  )

我们可以可视化数据集中的一些图像及其对应的类别标签：

In [ ]:
samples = [[] for _ in range(10)]
for image, label in dataset["test"]:
  if len(samples[label]) < 4:
    samples[label].append(image)

plt.figure(figsize=(20, 9))
for index in range(40):
  label = index % 10
  image = samples[label][index // 10]

  # Convert from CHW to HWC for visualization
  image = image.permute(1, 2, 0)

  # Convert from class index to class name
  label = dataset["test"].classes[label]

  # Visualize the image
  plt.subplot(4, 10, index + 1)
  plt.imshow(image)
  plt.title(label)
  plt.axis("off")
plt.show()

要训练神经网络，我们需要批量输入数据。我们创建批次大小为512的数据加载器：

In [ ]:
dataflow = {}
for split in ['train', 'test']:
  dataflow[split] = DataLoader(
    dataset[split],
    batch_size=512,
    shuffle=(split == 'train'),
    num_workers=0,
    pin_memory=True,
  )

我们可以打印训练数据加载器中的数据类型和形状：

In [ ]:
for inputs, targets in dataflow["train"]:
  print("[inputs] dtype: {}, shape: {}".format(inputs.dtype, inputs.shape))
  print("[targets] dtype: {}, shape: {}".format(targets.dtype, targets.shape))
  break

### 模型

在本教程中，我们将使用[VGG-11](https://arxiv.org/abs/1409.1556)的一个变体（具有更少的下采样和更小的分类器）作为我们的模型。

In [ ]:
class VGG(nn.Module):
  ARCH = [64, 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M']

  def __init__(self) -> None:
    super().__init__()

    layers = []
    counts = defaultdict(int)

    def add(name: str, layer: nn.Module) -> None:
      layers.append((f"{name}{counts[name]}", layer))
      counts[name] += 1

    in_channels = 3
    for x in self.ARCH:
      if x != 'M':
        # conv-bn-relu
        add("conv", nn.Conv2d(in_channels, x, 3, padding=1, bias=False))
        add("bn", nn.BatchNorm2d(x))
        add("relu", nn.ReLU(True))
        in_channels = x
      else:
        # maxpool
        add("pool", nn.MaxPool2d(2))

    self.backbone = nn.Sequential(OrderedDict(layers))
    self.classifier = nn.Linear(512, 10)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    # backbone: [N, 3, 32, 32] => [N, 512, 2, 2]
    x = self.backbone(x)

    # avgpool: [N, 512, 2, 2] => [N, 512]
    x = x.mean([2, 3])

    # classifier: [N, 512] => [N, 10]
    x = self.classifier(x)
    return x

model = VGG().cuda()

其主干由八个`conv-bn-relu`块和四个`maxpool`交错组成，将特征图下采样2^4 = 16倍：

In [ ]:
print(model.backbone)

特征图池化后，其分类器通过一个线性层预测最终输出：

In [ ]:
print(model.classifier)

由于本课程关注效率，我们将检查其模型大小和（理论）计算成本。


* 模型大小可通过可训练参数的数量来估计：

In [ ]:
num_params = 0
for param in model.parameters():
  if param.requires_grad:
    num_params += param.numel()
print("#Params:", num_params)

* 计算成本可通过[乘累加操作数（MACs）](https://en.wikipedia.org/wiki/Multiply–accumulate_operation)的数量来估计，使用[TorchProfile](https://github.com/zhijian-liu/torchprofile)：

In [ ]:
num_macs = profile_macs(model, torch.zeros(1, 3, 32, 32).cuda())
print("#MACs:", num_macs)

该模型有920万参数，推理需要6.06亿MACs。我们将在接下来的几个实验中共同努力提高其效率。

### 优化

由于我们在处理分类问题，我们将使用[交叉熵](https://en.wikipedia.org/wiki/Cross_entropy)作为损失函数来优化模型：

In [ ]:
criterion = nn.CrossEntropyLoss()

优化将使用[随机梯度下降（SGD）](https://en.wikipedia.org/wiki/Stochastic_gradient_descent)结合[动量](https://en.wikipedia.org/wiki/Stochastic_gradient_descent#Momentum)进行：

In [ ]:
optimizer = SGD(
  model.parameters(),
  lr=0.4,
  momentum=0.9,
  weight_decay=5e-4,
)

学习率将使用以下调度器进行调节（改编自[本博客系列](https://myrtle.ai/learn/how-to-train-your-resnet/)）：

In [ ]:
num_epochs = 20
steps_per_epoch = len(dataflow["train"])

# Define the piecewise linear scheduler
lr_lambda = lambda step: np.interp(
  [step / steps_per_epoch],
  [0, num_epochs * 0.3, num_epochs],
  [0, 1, 0]
)[0]

# Visualize the learning rate schedule
steps = np.arange(steps_per_epoch * num_epochs)
plt.plot(steps, [lr_lambda(step) * 0.4 for step in steps])
plt.xlabel("Number of Steps")
plt.ylabel("Learning Rate")
plt.grid("on")
plt.show()

scheduler = LambdaLR(optimizer, lr_lambda)

### 训练

我们首先定义训练函数，该函数对模型进行一个epoch的优化（*即*，遍历一次训练集）：

In [ ]:
def train(
  model: nn.Module,
  dataflow: DataLoader,
  criterion: nn.Module,
  optimizer: Optimizer,
  scheduler: LambdaLR,
) -> None:
  model.train()

  for inputs, targets in tqdm(dataflow, desc='train', leave=False):
    # Move the data from CPU to GPU
    inputs = inputs.cuda()
    targets = targets.cuda()

    # Reset the gradients (from the last iteration)
    optimizer.zero_grad()

    # Forward inference
    outputs = model(inputs)
    loss = criterion(outputs, targets)

    # Backward propagation
    loss.backward()

    # Update optimizer and LR scheduler
    optimizer.step()
    scheduler.step()

然后我们定义评估函数，该函数计算测试集上的指标（*即*，本实验中的准确率）：

In [ ]:
@torch.inference_mode()
def evaluate(
  model: nn.Module,
  dataflow: DataLoader
) -> float:
  model.eval()

  num_samples = 0
  num_correct = 0

  for inputs, targets in tqdm(dataflow, desc="eval", leave=False):
    # Move the data from CPU to GPU
    inputs = inputs.cuda()
    targets = targets.cuda()

    # Inference
    outputs = model(inputs)

    # Convert logits to class indices
    outputs = outputs.argmax(dim=1)

    # Update metrics
    num_samples += targets.size(0)
    num_correct += (outputs == targets).sum()

  return (num_correct / num_samples * 100).item()

有了训练和评估函数，我们终于可以开始训练模型了！这将需要大约10分钟。

In [ ]:
for epoch_num in tqdm(range(1, num_epochs + 1)):
  train(model, dataflow["train"], criterion, optimizer, scheduler)
  metric = evaluate(model, dataflow["test"])
  print(f"epoch {epoch_num}:", metric)

如果一切顺利，你训练好的模型应该能够达到>92.5\%的准确率！

### 可视化

我们可以可视化模型的预测结果，看看模型的实际表现如何：

In [ ]:
plt.figure(figsize=(20, 10))
for index in range(40):
  image, label = dataset["test"][index]

  # Model inference
  model.eval()
  with torch.inference_mode():
    pred = model(image.unsqueeze(dim=0).cuda())
    pred = pred.argmax(dim=1)

  # Convert from CHW to HWC for visualization
  image = image.permute(1, 2, 0)

  # Convert from class indices to class names
  pred = dataset["test"].classes[pred]
  label = dataset["test"].classes[label]

  # Visualize the image
  plt.subplot(4, 10, index + 1)
  plt.imshow(image)
  plt.title(f"pred: {pred}" + "\n" + f"label: {label}")
  plt.axis("off")
plt.show()